# Identifier length analysis

Parse the repository's Python sources without importing or running them. Exclude paths matched by `.gitignore` (including nested rules), `.git/info/exclude`, and coverage omit settings in `pyproject.toml`; also exclude `__init__.py` and path components containing `synthesized`, `synthezised`, or `synthesised`.

An identifier is a name represented in the Python syntax tree: references, definitions, parameters, attributes, imports and aliases, keyword arguments, exception bindings, pattern bindings, and type parameters. Dotted attributes and import names are split into independent names. For example, `identifier.attribute` contributes `identifier` and `attribute`. Keywords, comments, string contents (including quoted annotations), and wildcard imports are excluded.

Both **unique names** (each spelling counted once) and **occurrences** (every appearance counted) are analyzed. Character counts include underscores and count Unicode code points in the parser's normalized identifiers. Run all cells using this repository's Python environment; the kernel can start anywhere inside the repository.


In [1]:
#ruff: file-ignore[undocumented-public-function, suspicious-subprocess-import, start-process-with-partial-path]
#ruff: file-ignore[raise-vanilla-args, raw-string-in-exception]
#ruff: file-ignore[too-many-return-statements, too-many-boolean-expressions]
from __future__ import annotations

from astToolkit import Be, NodeTourist
from astToolkit.filesystem import parsePathFilename2astModule
from coverage.files import GlobMatcher
from functools import partial
from IPython.display import HTML
from itertools import chain
from pathlib import Path, PurePosixPath
from subprocess import run
from typing import TYPE_CHECKING
import IPython
import polars
import re
import tomllib

if TYPE_CHECKING:
	import ast

#=SIN= Filesystem selection is local because kitFilesystem has no repository-source inventory operation.
pathRepository: Path = Path(run(
	["git", "rev-parse", "--show-toplevel"],
	check=True, capture_output=True, text=True, encoding="utf-8",
).stdout.strip())

polars.Config.set_float_precision(4)
pathRepository


WindowsPath('C:/apps/mapFolding')

In [2]:
def getExclusionReason(
	relativePathFilename: str,
	pathRepository: Path,
	gitIgnoredPathFilenames: frozenset[str],
	coverageMatcher: GlobMatcher,
) -> str:
	if relativePathFilename in gitIgnoredPathFilenames:
		return "Git ignore"
	if PurePosixPath(relativePathFilename).name == "__init__.py":
		return "__init__.py"
	if re.search(r"(?i)(?:synthesized|synthezised|synthesised)", relativePathFilename):
		return "Synthesized"
	if coverageMatcher.match(relativePathFilename) or coverageMatcher.match(str(pathRepository / relativePathFilename)):
		return "Coverage omit"
	if not (pathRepository / relativePathFilename).is_file():
		return "Missing from working tree"
	return "Included"

#=SIN= The candidate tuple is required because Git's stdin filter and the source inventory consume the same discovery.
relativePathFilenamesCandidate: tuple[str, ...] = tuple(sorted(set(filter(None, run(
	["git", "-c", "core.excludesFile=", "ls-files", "--cached", "--others", "--exclude-standard", "-z", "--", "*.py"],
	cwd=pathRepository, check=True, capture_output=True, text=True, encoding="utf-8",
).stdout.split("\0")))))

gitIgnoreInspection = run(
	["git", "-c", "core.excludesFile=", "check-ignore", "--no-index", "--stdin", "-z"],
	check=False, input="\0".join(relativePathFilenamesCandidate) + "\0",
	cwd=pathRepository, capture_output=True, text=True, encoding="utf-8",
)
if gitIgnoreInspection.returncode not in {0, 1}:
	gitIgnoreInspection.check_returncode()

gitIgnoredPathFilenames: frozenset[str] = frozenset(filter(None, gitIgnoreInspection.stdout.split("\0")))

coverageSettings = tomllib.loads(
	(pathRepository / "pyproject.toml").read_text(encoding="utf-8")
).get("tool", {}).get("coverage", {})

coverageMatcher: GlobMatcher = GlobMatcher(chain(
	coverageSettings.get("omit", ()),
	coverageSettings.get("run", {}).get("omit", ()),
	coverageSettings.get("report", {}).get("omit", ()),
))

sourceInventory: polars.DataFrame = polars.DataFrame({
	"relativePathFilename": relativePathFilenamesCandidate,
	"exclusionReason": tuple(map(
		partial(
			getExclusionReason,
			pathRepository=pathRepository,
			gitIgnoredPathFilenames=gitIgnoredPathFilenames,
			coverageMatcher=coverageMatcher,
		),
		relativePathFilenamesCandidate,
	)),
}, schema={"relativePathFilename": polars.String, "exclusionReason": polars.String})

includedPythonSources: polars.DataFrame = sourceInventory.lazy().filter(
	polars.col("exclusionReason") == "Included"
).select("relativePathFilename").collect()

if includedPythonSources.is_empty():
	raise ValueError("No Python sources remain after applying the identifier-analysis exclusions.")

with polars.Config(tbl_rows=100, tbl_cols=10):
	IPython.display.display(sourceInventory.lazy().group_by("exclusionReason").agg(
		polars.len().alias("sourceCount")
	).sort("exclusionReason").collect())
	IPython.display.display(includedPythonSources)


exclusionReason,sourceCount
str,u32
"""Coverage omit""",92
"""Included""",83
"""Synthesized""",20
"""__init__.py""",35


relativePathFilename
str
"""mapFolding/_e/_beDRY.py"""
"""mapFolding/_e/_disaggregation.…"
"""mapFolding/_e/algorithms/const…"
"""mapFolding/_e/algorithms/elimi…"
"""mapFolding/_e/algorithms/elimi…"
"""mapFolding/_e/algorithms/iff.p…"
"""mapFolding/_e/algorithms/inser…"
"""mapFolding/_e/algorithms/inser…"
"""mapFolding/_e/algorithms/nisha…"


## Parse names

Git supplies tracked sources and untracked, nonignored sources. Tracked paths are checked again with `git check-ignore --no-index`, so tracked sources that match an ignore rule are excluded too. Global Git ignore settings are disabled for this scan. `sourceInventory` records each discovered path and its exclusion reason; ignored untracked paths are omitted by Git during discovery.

Parsing errors stop execution instead of silently dropping sources. The AST identifier fields follow [Python's abstract grammar](https://docs.python.org/3/library/ast.html#abstract-grammar).


In [3]:
def getNodeIdentifierNames(astNode: ast.AST) -> tuple[str, ...]:
	if Be.Name(astNode):
		return (astNode.id,)
	if Be.Attribute(astNode):
		return (astNode.attr,)
	if Be.arg(astNode) or Be.keyword(astNode):
		return (astNode.arg or "",)
	if Be.alias(astNode):
		return tuple(chain(astNode.name.split("."), (astNode.asname or "",)))
	if Be.ImportFrom(astNode):
		return tuple((astNode.module or "").split("."))
	if Be.Global(astNode) or Be.Nonlocal(astNode):
		return tuple(astNode.names)
	if Be.MatchClass(astNode):
		return tuple(astNode.kwd_attrs)
	if Be.MatchMapping(astNode):
		return (astNode.rest or "",)
	if (
		Be.FunctionDef(astNode)
		or Be.AsyncFunctionDef(astNode)
		or Be.ClassDef(astNode)
		or Be.ExceptHandler(astNode)
		or Be.MatchAs(astNode)
		or Be.MatchStar(astNode)
		or Be.TypeVar(astNode)
		or Be.ParamSpec(astNode)
		or Be.TypeVarTuple(astNode)
	):
		return (astNode.name or "",)
	return ()

def getIdentifierOccurrences(relativePathFilename: str, pathRepository: Path) -> polars.LazyFrame:
	#=SIN= An AST-node collection is required because NodeTourist only returns its last captured match.
	list_astNodes: list[ast.AST] = []
	NodeTourist(Be.AST, list_astNodes.append).visit(
		parsePathFilename2astModule(pathRepository / relativePathFilename, filename=relativePathFilename)
	)
	return polars.DataFrame({
		"identifier": tuple(filter(
			str.isidentifier,
			chain.from_iterable(map(getNodeIdentifierNames, list_astNodes)),
		)),
	}, schema={"identifier": polars.String}).lazy().with_columns(
		polars.lit(relativePathFilename).alias("relativePathFilename"),
		polars.col("identifier").str.len_chars().alias("characterCount"),
	)

identifierOccurrences: polars.DataFrame = polars.concat(map(
	partial(getIdentifierOccurrences, pathRepository=pathRepository),
	includedPythonSources.get_column("relativePathFilename"),
)).collect()

if identifierOccurrences.is_empty():
	raise ValueError("The included Python sources contain no identifier names.")

identifierInventory: polars.DataFrame = identifierOccurrences.lazy().group_by(
	"identifier", "characterCount"
).agg(
	polars.len().alias("occurrenceCount"),
	polars.col("relativePathFilename").n_unique().alias("sourceCount"),
).sort("identifier").collect()

IPython.display.display(polars.DataFrame({
	"parsedSourceCount": [includedPythonSources.height],
	"uniqueNameCount": [identifierInventory.height],
	"occurrenceCount": [identifierOccurrences.height],
}))


parsedSourceCount,uniqueNameCount,occurrenceCount
i64,i64,i64
83,2478,34040


## Character-count statistics

Standard deviation is the **population** standard deviation (`ddof=0`) because the scan observes the full selected codebase. The ±1σ and ±2σ intervals are mean ± the corresponding multiple of standard deviation. These intervals describe lengths; they do not assume a normal distribution and can have a negative lower bound.

Quintile cutoffs are the 20th, 40th, 60th, and 80th percentiles, with linear interpolation. The minimum and maximum complete the five intervals. Repeated integer lengths can produce tied cutoffs; interpolated cutoffs need not be integers. See [Polars standard deviation](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.std.html) and [quantiles](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.quantile.html).


In [4]:
identifierLengthPopulations: polars.LazyFrame = polars.concat((
	identifierInventory.lazy().select("identifier", "characterCount").with_columns(
		polars.lit("Unique names").alias("population")
	),
	identifierOccurrences.lazy().select("identifier", "characterCount").with_columns(
		polars.lit("Occurrences").alias("population")
	),
))

identifierLengthStatistics: polars.DataFrame = identifierLengthPopulations.group_by("population").agg(
	polars.len().alias("identifierCount"),
	polars.col("characterCount").min().alias("minimum"),
	polars.col("characterCount").max().alias("maximum"),
	polars.col("characterCount").mean().alias("mean"),
	polars.col("characterCount").median().alias("median"),
	polars.col("characterCount").std(ddof=0).alias("standardDeviation"),
).with_columns(
	(polars.col("mean") - polars.col("standardDeviation")).alias("meanMinus1StandardDeviation"),
	(polars.col("mean") + polars.col("standardDeviation")).alias("meanPlus1StandardDeviation"),
	(polars.col("mean") - 2 * polars.col("standardDeviation")).alias("meanMinus2StandardDeviations"),
	(polars.col("mean") + 2 * polars.col("standardDeviation")).alias("meanPlus2StandardDeviations"),
).sort("population", descending=True).collect()

with polars.Config(tbl_rows=30, tbl_cols=10):
	IPython.display.display(identifierLengthStatistics.lazy().unpivot(
		index="population", variable_name="statistic", value_name="characterCount"
	).collect())


population,statistic,characterCount
str,str,f64
"""Unique names""","""identifierCount""",2478.0000
"""Unique names""","""minimum""",1.0000
"""Unique names""","""maximum""",53.0000
"""Unique names""","""mean""",12.4661
"""Unique names""","""median""",11.0000
"""Unique names""","""standardDeviation""",7.1211
"""Unique names""","""meanMinus1StandardDeviation""",5.3450
"""Unique names""","""meanPlus1StandardDeviation""",19.5872
"""Unique names""","""meanMinus2StandardDeviations""",-1.7760


In [5]:
identifierLengthQuintiles: polars.DataFrame = identifierLengthPopulations.group_by("population").agg(
	polars.col("characterCount").min().alias("minimum"),
	polars.col("characterCount").quantile(0.2, interpolation="linear").alias("quintile20"),
	polars.col("characterCount").quantile(0.4, interpolation="linear").alias("quintile40"),
	polars.col("characterCount").quantile(0.6, interpolation="linear").alias("quintile60"),
	polars.col("characterCount").quantile(0.8, interpolation="linear").alias("quintile80"),
	polars.col("characterCount").max().alias("maximum"),
).sort("population", descending=True).collect()

with polars.Config(tbl_rows=20, tbl_cols=10):
	IPython.display.display(identifierLengthQuintiles.lazy().unpivot(
		index="population", variable_name="cutoff", value_name="characterCount"
	).collect())


population,cutoff,characterCount
str,str,f64
"""Unique names""","""minimum""",1.0000
"""Unique names""","""quintile20""",7.0000
"""Unique names""","""quintile40""",10.0000
"""Unique names""","""quintile60""",13.0000
"""Unique names""","""quintile80""",18.0000
"""Unique names""","""maximum""",53.0000
"""Occurrences""","""minimum""",1.0000
"""Occurrences""","""quintile20""",4.0000
"""Occurrences""","""quintile40""",6.0000


In [6]:
identifierLengthIntervalCoverage: polars.DataFrame = identifierLengthPopulations.join(
	identifierLengthStatistics.lazy(), on="population"
).group_by("population").agg(
	polars.col("characterCount").is_between(
		polars.col("meanMinus1StandardDeviation"),
		polars.col("meanPlus1StandardDeviation"),
		closed="both",
	).sum().alias("countWithin1StandardDeviation"),
	(100 * polars.col("characterCount").is_between(
		polars.col("meanMinus1StandardDeviation"),
		polars.col("meanPlus1StandardDeviation"),
		closed="both",
	).mean()).alias("percentWithin1StandardDeviation"),
	polars.col("characterCount").is_between(
		polars.col("meanMinus2StandardDeviations"),
		polars.col("meanPlus2StandardDeviations"),
		closed="both",
	).sum().alias("countWithin2StandardDeviations"),
	(100 * polars.col("characterCount").is_between(
		polars.col("meanMinus2StandardDeviations"),
		polars.col("meanPlus2StandardDeviations"),
		closed="both",
	).mean()).alias("percentWithin2StandardDeviations"),
).sort("population", descending=True).collect()

with polars.Config(tbl_cols=10):
	IPython.display.display(identifierLengthIntervalCoverage)


population,countWithin1StandardDeviation,percentWithin1StandardDeviation,countWithin2StandardDeviations,percentWithin2StandardDeviations
str,u32,f64,u32,f64
"""Unique names""",1785,72.0339,2376,95.8838
"""Occurrences""",26726,78.5135,32720,96.1222


## Distribution and complete inventory

The distribution counts unique names and occurrences at each character count. The complete alphabetized inventory is expandable below. `identifierInventory` contains every unique name, its character count, its occurrence count, and the number of sources using it. `identifierOccurrences` retains every occurrence and its source path.


In [7]:
identifierLengthDistribution: polars.DataFrame = identifierInventory.lazy().group_by("characterCount").agg(
	polars.len().alias("uniqueNameCount"),
	polars.col("occurrenceCount").sum().alias("occurrenceCount"),
).sort("characterCount").collect()

with polars.Config(tbl_rows=-1, tbl_cols=10):
	IPython.display.display(identifierLengthDistribution)
	IPython.display.display(HTML(
		f"<details><summary>Complete inventory: {identifierInventory.height:,} unique identifier names</summary>"
		f"{identifierInventory._repr_html_()}</details>"
	))


characterCount,uniqueNameCount,occurrenceCount
u32,u32,u32
1,21,1673
2,27,596
3,72,3307
4,111,3906
5,117,3494
6,135,1887
7,157,1543
8,195,2693
9,151,1657


identifier,characterCount,occurrenceCount,sourceCount
str,u32,u32,u32
"""A000136""",7,8,5
"""A000560""",7,9,3
"""A000682""",7,6,5
"""A001010""",7,10,3
"""A001011""",7,5,3
"""A005315""",7,17,5
"""A005315of0""",10,2,1
"""A005315of1""",10,4,2
"""A005315of10""",11,4,2
